[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_08/ATS_ch8_mgarch/ATS_ch8_mgarch.ipynb)

# ATS_ch8_mgarch

Multivariate GARCH: parameter counts of VEC, BEKK and DCC; DCC against cDCC (Aielli 2013) by simulation; global minimum-variance portfolios of 25 US stocks and ETFs with sample, Ledoit-Wolf linear, analytical nonlinear shrinkage (Ledoit and Wolf 2020), DCC and DCC-NL (Engle, Ledoit and Wolf 2019) covariance matrices; realised (5-minute) and DCC correlations of Bitcoin and Ether.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 8: Advanced volatility modelling: realised measures, HAR and multivariate GARCH. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_8'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import time

from scipy import special

from scipy.signal import lfilter

SEED = 2026

REAL_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/realized/'

REAL_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                 if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')

OMI_NAMES = {'.SPX': 'S&P 500', '.GDAXI': 'DAX', '.FTSE': 'FTSE 100', '.N225': 'Nikkei 225', '.STOXX50E': 'Euro Stoxx 50', '.FCHI': 'CAC 40'}

MU1 = np.sqrt(2 / np.pi)

HT_C = np.pi ** 2 / 4 + np.pi - 5

SIM = {'n': 23400, 'mean_iv': 1.0, 'sd_slow': 0.75, 'sd_fast': 0.45, 'hl_slow': 60.0, 'hl_fast': 2.0, 'rho': -0.6, 'lam': 0.08, 'sd_jump': 0.8, 'omega': 0.004}

HAR_WIN = 1000

GMV_ASSETS = ['AAPL.US', 'AMZN.US', 'BAC.US', 'C.US', 'GOOGL.US', 'GS.US', 'JPM.US', 'META.US', 'MS.US', 'MSFT.US', 'MSTR.US', 'NVDA.US', 'TSLA.US', 'WFC.US', 'SPY.US', 'QQQ.US', 'RSP.US']

def save(name, save_it=True):
    st.check_no_grey(plt.gcf())
    if save_it:
        st.save_fig(name)
    else:
        plt.show()
        plt.close()


def read_realized(fname, **kw):
    """A file of data/realized: local copy of the course data, otherwise the ATS repository on GitHub."""
    p = os.path.join(REAL_DIR, fname) if REAL_DIR else ''
    return pd.read_csv(p if p and os.path.exists(p) else REAL_RAW + fname, **kw)


def omi(symbol='.SPX'):
    """Oxford-Man realized library v0.3 (Heber, Lunde, Shephard and Sheppard 2009): one index, daily, returns in %
    and variances in %^2; days with a non-positive measure are dropped."""
    d = read_omi()
    d = d[d['symbol'] == symbol].set_index('date').sort_index()
    out = pd.DataFrame({'rv': 1e4 * d['rv5'], 'rv_ss': 1e4 * d['rv5_ss'], 'bv': 1e4 * d['bv'],
                        'medrv': 1e4 * d['medrv'], 'rk': 1e4 * d['rk_parzen'], 'rsv': 1e4 * d['rsv'],
                        'oc': 100 * d['open_to_close'], 'nobs': d['nobs']})
    return out[(out[['rv', 'bv', 'rk', 'medrv']] > 0).all(axis=1)].dropna()


def binance_daily(coin='btc'):
    """Our daily realised measures of Bitcoin or Ether from Binance one-minute prices (UTC days, %, %^2)."""
    d = read_realized('binance_daily_realized.csv', parse_dates=['date'], index_col='date')
    c = [x for x in d.columns if x.startswith(coin + '_')]
    out = d[c].rename(columns=lambda x: x[len(coin) + 1:]).dropna()
    out = out[(out[['rv5', 'bv5', 'rk1']] > 0).all(axis=1)]
    if 'rcov5' in d.columns:
        out['rcov5'] = d['rcov5'].reindex(out.index)
    return out


def returns(name, start='2000-01-01', end=None):
    """Daily log returns in %."""
    p = load_close(name, start=start, end=end) if end else load_close(name, start=start)
    return (100 * np.log(p).diff()).dropna()


def garch_var(e, omega, alpha, beta, gamma=0.0, s0=None):
    """Conditional variances of a (GJR-)GARCH(1,1) by a linear filter:
    s_t = omega + (alpha + gamma 1[e_{t-1} < 0]) e_{t-1}^2 + beta s_{t-1}, s_1 = s0 (sample variance)."""
    e = np.asarray(e, float)
    s0 = np.var(e) if s0 is None else s0
    x = np.empty_like(e)
    x[0] = s0
    x[1:] = omega + (alpha + gamma * (e[:-1] < 0)) * e[:-1] ** 2
    return lfilter([1.0], [1.0, -beta], x)


def t_logpdf_std(z, nu):
    """Log density of a Student-t standardised to unit variance."""
    return (special.gammaln((nu + 1) / 2) - special.gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2))
            - (nu + 1) / 2 * np.log1p(z ** 2 / (nu - 2)))


def garch_lt(theta, e, dist='normal', gjr=False):
    """Per-observation log-likelihood of a (GJR-)GARCH(1,1) with Gaussian or standardised Student-t innovations."""
    omega, alpha, beta = theta[:3]
    gamma = theta[3] if gjr else 0.0
    s = garch_var(e, omega, alpha, beta, gamma)
    if dist == 'normal':
        return -0.5 * (np.log(2 * np.pi) + np.log(s) + e ** 2 / s)
    nu = theta[-1]
    return t_logpdf_std(e / np.sqrt(s), nu) - 0.5 * np.log(s)


def num_scores(f, theta, h=1e-5):
    """Numerical per-observation scores (T x k) of a per-observation log-likelihood f(theta)."""
    theta = np.asarray(theta, float)
    out = []
    for j in range(len(theta)):
        d = np.zeros_like(theta)
        d[j] = h * max(1.0, abs(theta[j]))
        out.append((f(theta + d) - f(theta - d)) / (2 * d[j]))
    return np.column_stack(out)


def num_hessian(f, theta, h=1e-4):
    """Numerical Hessian of the summed log-likelihood, from differences of the summed scores."""
    theta = np.asarray(theta, float)
    k = len(theta)
    H = np.zeros((k, k))
    for j in range(k):
        d = np.zeros(k)
        d[j] = h * max(1.0, abs(theta[j]))
        H[:, j] = (num_scores(f, theta + d).sum(0) - num_scores(f, theta - d).sum(0)) / (2 * d[j])
    return 0.5 * (H + H.T)


def sandwich(f, theta):
    """Hessian-based, outer-product and Bollerslev-Wooldridge (sandwich) standard errors."""
    S = num_scores(f, theta)
    H = num_hessian(f, theta)
    Hi = np.linalg.inv(-H)
    I = S.T @ S
    return dict(se_h=np.sqrt(np.diag(Hi)), se_opg=np.sqrt(np.diag(np.linalg.inv(I))),
                se_bw=np.sqrt(np.diag(Hi @ I @ Hi)), cov_bw=Hi @ I @ Hi)


def fit_garch(e, dist='normal', gjr=False, se=True):
    """(Q)ML of a (GJR-)GARCH(1,1) on demeaned returns e (in %)."""
    e = np.asarray(e, float)
    v = np.var(e)
    x0 = [0.05 * v, 0.08, 0.90] + ([0.05] if gjr else []) + ([8.0] if dist == 't' else [])
    bnds = [(1e-8, 10 * v), (1e-6, 0.6), (0.0, 0.9999)] + ([(-0.3, 0.6)] if gjr else []) + ([(2.2, 200)] if dist == 't' else [])

    def nll(th):
        a = th[1] + (0.5 * th[3] if gjr else 0) + th[2]
        if a >= 0.99999 or (gjr and th[1] + th[3] < 0):
            return 1e10
        val = -garch_lt(th, e, dist, gjr).sum()
        return val if np.isfinite(val) else 1e10
    best = None
    for b0 in (0.90, 0.80, 0.95):
        x = list(x0)
        x[2] = b0
        r = optimize.minimize(nll, x, method='L-BFGS-B', bounds=bnds)
        if best is None or r.fun < best.fun:
            best = r
    th = best.x
    out = dict(theta=th, ll=-best.fun, s=garch_var(e, th[0], th[1], th[2], th[3] if gjr else 0.0))
    if se:
        out.update(sandwich(lambda t: garch_lt(t, e, dist, gjr), th))
    return out


def n_params(N):
    k = N * (N + 1) // 2
    return dict(vec=k + 2 * k * k, bekk=k + 2 * N * N, dbekk=k + 2 * N, sbekk=k + 2, dcc=3 * N + 2 + N * (N - 1) // 2)


def dcc_loglik(ab, Z, S, corrected=False, return_R=False):
    """Correlation part of the DCC (Engle 2002) or cDCC (Aielli 2013) log-likelihood for standardised residuals Z
    (T x N) and target S. DCC: Q_t = (1-a-b) S + a z z' + b Q_{t-1}; cDCC: z replaced by Q*_{t-1}^{1/2} z."""
    a, b = ab
    T, N = Z.shape
    Q = S.copy()
    ll = 0.0
    Rs = [] if return_R else None
    for t in range(T):
        d = np.sqrt(np.diag(Q))
        R = Q / np.outer(d, d)
        if return_R:
            Rs.append(R)
        z = Z[t]
        c = np.linalg.cholesky(R)
        w = np.linalg.solve(c, z)
        ll += -np.sum(np.log(np.diag(c))) - 0.5 * (w @ w - z @ z)
        zz = d * z if corrected else z
        Q = (1 - a - b) * S + a * np.outer(zz, zz) + b * Q
    return (ll, Rs) if return_R else ll


def cdcc_target(Z, a, b):
    """Aielli (2013) target: the sample covariance of Q*_{t}^{1/2} z_t, computed along the cDCC recursion."""
    T, N = Z.shape
    S = np.corrcoef(Z.T)
    for _ in range(3):
        Q = S.copy()
        acc = np.zeros((N, N))
        for t in range(T):
            d = np.sqrt(np.diag(Q))
            zz = d * Z[t]
            acc += np.outer(zz, zz)
            Q = (1 - a - b) * S + a * np.outer(zz, zz) + b * Q
        S = acc / T
        dd = np.sqrt(np.diag(S))
        S = S / np.outer(dd, dd)
    return S


def fit_dcc(Z, S=None, corrected=False):
    S = np.corrcoef(Z.T) if S is None else S

    def nll(ab):
        if ab[0] <= 0 or ab[1] <= 0 or ab[0] + ab[1] >= 0.999:
            return 1e10
        if corrected:
            return -dcc_loglik(ab, Z, cdcc_target(Z, *ab), corrected=True)
        return -dcc_loglik(ab, Z, S)
    best = None
    for x0 in ([0.03, 0.95], [0.08, 0.85]):
        r = optimize.minimize(nll, x0, method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-5, maxiter=400))
        if best is None or r.fun < best.fun:
            best = r
    return best.x, -best.fun


def simulate_cdcc(T, a, b, rho, rng):
    S = np.array([[1, rho], [rho, 1]])
    Q = S.copy()
    Z = np.empty((T, 2))
    for t in range(T):
        d = np.sqrt(np.diag(Q))
        R = Q / np.outer(d, d)
        z = np.linalg.cholesky(R) @ rng.standard_normal(2)
        Z[t] = z
        zz = d * z
        Q = (1 - a - b) * S + a * np.outer(zz, zz) + b * Q
    return Z


def lw_linear(X):
    """Ledoit-Wolf (2004) linear shrinkage towards the scaled identity (X demeaned, T x N)."""
    T, N = X.shape
    S = X.T @ X / T
    mu = np.trace(S) / N
    d2 = np.sum((S - mu * np.eye(N)) ** 2) / N
    b2 = sum(np.sum((np.outer(x, x) - S) ** 2) for x in X) / N / T ** 2
    b2 = min(b2, d2)
    return b2 / d2 * mu * np.eye(N) + (d2 - b2) / d2 * S


def nl_shrink(X):
    """Analytical nonlinear shrinkage of Ledoit and Wolf (2020, Annals of Statistics), X demeaned (T x N), N < T."""
    T, N = X.shape
    S = X.T @ X / T
    lam, U = np.linalg.eigh(S)
    lam = np.maximum(lam, 1e-12)
    h = T ** (-1 / 3)
    L = lam[None, :]
    H = h * L
    x = (lam[:, None] - L) / H
    ftil = np.mean(np.maximum(1 - x ** 2 / 5, 0) / H, axis=1) * 3 / (4 * np.sqrt(5))
    ad = np.abs(x)
    with np.errstate(divide='ignore', invalid='ignore'):
        Hf = ((-3 / (10 * np.pi)) * x + (3 / (4 * np.sqrt(5) * np.pi)) * (1 - x ** 2 / 5)
              * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x))))
    Hf = np.where(ad == np.sqrt(5), (-3 / (10 * np.pi)) * x, Hf)
    Htil = np.mean(Hf / H, axis=1)
    c = N / T
    dtil = lam / ((np.pi * c * lam * ftil) ** 2 + (1 - c - np.pi * c * lam * Htil) ** 2)
    return (U * dtil) @ U.T


def garch_univariate_std(R, hold=1):
    """Univariate Gaussian GARCH(1,1) for each column: standardised residuals and the variance forecast averaged over
    the next `hold` days (hold = 1: the one-day-ahead variance)."""
    Z = np.empty_like(R)
    nxt = np.empty(R.shape[1])
    for j in range(R.shape[1]):
        e = R[:, j]
        th = fit_garch(e, se=False)['theta']
        s = garch_var(np.r_[e, 0.0], th[0], th[1], th[2])
        Z[:, j] = e / np.sqrt(s[:-1])
        p = th[1] + th[2]
        lr = th[0] / (1 - p)
        nxt[j] = lr + (s[-1] - lr) * np.mean(p ** np.arange(hold))
    return Z, nxt


def gmv(Sig):
    w = np.linalg.solve(Sig, np.ones(len(Sig)))
    return w / w.sum()


def corr_shrunk(X, kind):
    """Shrink the correlation matrix of the columns of X (linear LW or nonlinear), then rescale by the sample s.d."""
    sd = X.std(0)
    Z = (X - X.mean(0)) / sd
    C = lw_linear(Z) if kind == 'lw' else nl_shrink(Z)
    d = np.sqrt(np.diag(C))
    return np.outer(sd, sd) * C / np.outer(d, d)


def fig_dcc_sim(save_it=True, reps=100, T=2000, a=0.05, b=0.93, rho=0.5):
    """Aielli (2013): DCC and cDCC estimators of (a, b) on data simulated from a cDCC model (bivariate, unit variances)."""
    rng = np.random.default_rng(SEED + 5)
    est = {'DCC': [], 'cDCC': []}
    for _ in range(reps):
        Z = simulate_cdcc(T, a, b, rho, rng)
        est['DCC'].append(fit_dcc(Z)[0])
        est['cDCC'].append(fit_dcc(Z, corrected=True)[0])
    est = {k: np.array(v) for k, v in est.items()}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
    for ax, j, nm, tv in zip(axes, (0, 1), ('a', 'b'), (a, b)):
        bins = np.linspace(min(est['DCC'][:, j].min(), est['cDCC'][:, j].min()),
                           max(est['DCC'][:, j].max(), est['cDCC'][:, j].max()), 30)
        ax.hist(est['DCC'][:, j], bins=bins, histtype='step', lw=1.6, color=st.IDAred, label='DCC estimator')
        ax.hist(est['cDCC'][:, j], bins=bins, histtype='step', lw=1.6, color=st.MainBlue, label='cDCC estimator')
        ax.axvline(tv, color=st.Forest, ls='--', lw=1.4, label='true value')
        ax.set_title(f'estimates of {nm}')
    st.fig_legend_bottom(fig, ncol=3, y=-0.02)
    plt.tight_layout()
    save('ats_ch8_dcc_sim', save_it)
    return dict(reps=reps, T=T, a=a, b=b, rho=rho,
                mean={k: v.mean(0).tolist() for k, v in est.items()}, sd={k: v.std(0).tolist() for k, v in est.items()},
                rmse={k: np.sqrt(((v - np.array([a, b])) ** 2).mean(0)).tolist() for k, v in est.items()})


def fig_gmv(save_it=True, win=1250, short=250, hold=21, start='2012-06-01'):
    """Out-of-sample global minimum-variance portfolios of 14 US stocks and 3 equity ETFs: sample covariance,
    Ledoit-Wolf linear and nonlinear shrinkage of the correlation matrix, DCC and DCC-NL (Engle, Ledoit and Wolf
    2019), 1/N; monthly rebalancing with DCC forecasts averaged over the 21-day holding period; the static
    estimators also with a short window (larger N/T)."""
    P = pd.concat([read_market(s)['adjusted_close'].rename(s) for s in GMV_ASSETS], axis=1).loc[start:]
    P = P[P.index.dayofweek < 5].dropna()
    R = (100 * np.log(P).diff()).dropna()
    Rv = R.values
    T, N = Rv.shape
    methods = ['1/N', 'sample', 'LW linear', 'NL shrinkage', 'DCC', 'DCC-NL', 'sample, short', 'NL, short']
    port = {m: [] for m in methods}
    ab = {'DCC': [], 'DCC-NL': []}
    conds = []
    dates = []
    for s in range(win, T - 1, hold):
        X = Rv[s - win:s]
        Xc = X - X.mean(0)
        Xs = Rv[s - short:s]
        e_ = min(s + hold, T)
        S_ = Xc.T @ Xc / win
        conds.append(np.linalg.cond(np.corrcoef(Xc.T)))
        Sig = {'sample': S_, 'LW linear': corr_shrunk(X, 'lw'), 'NL shrinkage': corr_shrunk(X, 'nl'),
               'sample, short': np.cov(Xs.T), 'NL, short': corr_shrunk(Xs, 'nl')}
        Z, nxt = garch_univariate_std(Xc, hold)
        Dn = np.sqrt(nxt)
        for nm in ('DCC', 'DCC-NL'):
            if nm == 'DCC':
                tgt = np.corrcoef(Z.T)
            else:
                C = nl_shrink(Z - Z.mean(0))
                dd = np.sqrt(np.diag(C))
                tgt = C / np.outer(dd, dd)
            (a, b), _ = fit_dcc(Z, tgt)
            ab[nm].append((a, b))
            Q = tgt.copy()
            for t in range(win):
                Q = (1 - a - b) * tgt + a * np.outer(Z[t], Z[t]) + b * Q
            Q = tgt + (Q - tgt) * np.mean((a + b) ** np.arange(hold))   # average forecast over the holding period
            dq = np.sqrt(np.diag(Q))
            Sig[nm] = np.outer(Dn, Dn) * Q / np.outer(dq, dq)
        W = {m: gmv(Sig[m]) for m in Sig}
        W['1/N'] = np.ones(N) / N
        for m in methods:
            port[m].append(Rv[s:e_] @ W[m])
        dates.append(R.index[s])
    out = {'N': N, 'T': T, 'first_oos': str(R.index[win].date()), 'last': str(R.index[-1].date()), 'n_rebal': len(dates),
           'win': win, 'short': short, 'first': str(R.index[0].date()), 'cond_med': float(np.median(conds))}
    sd = {m: float(np.sqrt(252) * np.concatenate(port[m]).std()) for m in methods}
    out['sd'] = sd
    out['ab'] = {k: np.mean(v, 0).tolist() for k, v in ab.items()}
    fig, ax = plt.subplots(figsize=(10.5, 4.0))
    x = np.arange(len(methods))
    cols = [st.Amber, st.IDAred, st.Orange, st.Teal, st.MainBlue, st.Forest, st.Crimson, st.Purple]
    ax.bar(x, [sd[m] for m in methods], color=cols)
    for i, m in enumerate(methods):
        ax.text(i, sd[m] + 0.2, f'{sd[m]:.2f}', ha='center', color=st.DarkText, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(', short', f'\n{short} days') for m in methods])
    ax.set_ylabel('out-of-sample s.d., % p.a.')
    ax.set_ylim(min(sd.values()) * 0.8, max(sd.values()) * 1.08)
    ax.bar([0], [0], color=st.MainBlue, label=f'GMV portfolios of {N} US equities and equity ETFs, monthly rebalancing; window {win} days unless stated')
    st.legend_outside_bottom(ax, ncol=1, y=-0.2)
    save('ats_ch8_gmv', save_it)
    return out


def fig_corr_crypto(save_it=True):
    """Daily realised correlation of Bitcoin and Ether (5-minute returns) against the DCC correlation of daily returns."""
    b = binance_daily('btc')
    e = binance_daily('eth')
    D = pd.concat([b['ret'].rename('b'), e['ret'].rename('e'), b['rv5'].rename('vb'), e['rv5'].rename('ve'),
                   b['rcov5'].rename('c')], axis=1).dropna()
    rc = D['c'] / np.sqrt(D['vb'] * D['ve'])
    Rr = D[['b', 'e']].values
    Rr = Rr - Rr.mean(0)
    Z, _ = garch_univariate_std(Rr)
    (a, bb), _ = fit_dcc(Z)
    _, Rs = dcc_loglik((a, bb), Z, np.corrcoef(Z.T), return_R=True)
    dcc = np.array([R[0, 1] for R in Rs])
    fig, ax = plt.subplots(figsize=(11, 4.0))
    ax.plot(D.index, rc.rolling(5).mean(), color=st.Purple, lw=0.7, label='realised correlation, 5-minute returns (5-day mean)')
    ax.plot(D.index, dcc, color=st.MainBlue, lw=1.4, label='DCC correlation, daily returns')
    ax.set_ylabel('correlation')
    st.legend_outside_bottom(ax, ncol=2, y=-0.15)
    save('ats_ch8_corr_crypto', save_it)
    return dict(T=len(D), a=float(a), b=float(bb), rc_mean=float(rc.mean()), dcc_mean=float(dcc.mean()),
                corr_rc_dcc=float(np.corrcoef(rc, dcc)[0, 1]), rc_q05=float(rc.quantile(0.05)),
                first=str(D.index[0].date()), last=str(D.index[-1].date()))

## Run

In [ ]:
print(n_params(25))
print(fig_dcc_sim(reps=10))
print(fig_gmv(hold=63))
print(fig_corr_crypto())

![ats_ch8_dcc_sim](./ats_ch8_dcc_sim.png)

Output: `ats_ch8_dcc_sim.pdf`

![ats_ch8_gmv](./ats_ch8_gmv.png)

Output: `ats_ch8_gmv.pdf`

![ats_ch8_corr_crypto](./ats_ch8_corr_crypto.png)

Output: `ats_ch8_corr_crypto.pdf`